# Fine-tuning based on Qwen3-1.7B with Style Vector Input

In [1]:
# Run this on Remote Jupyter Book.

import os

os.getcwd()
os.chdir("/root/OtakuLab")

## Define Training Set Format

In [2]:
from typing import Optional
from pathlib import Path
from typing_extensions import TypedDict
import json

DENSE_METRIC_KEYS = [
    "mdd",
    "verb_noun_ratio",
    "question_ratio",
    "imperative_ratio",
    "fragment_ratio",
    "marker_density",
]

class InstructionComponents(TypedDict):
    semantic_tags: list[str]
    structural_metrics: dict[str, float]
    signature_rules: list[str]

class DatasetItem(TypedDict):
    character: str
    neutral_sentence: str
    instruction_components: InstructionComponents
    thinking_process: str
    output: str

class DatasetStorage:
    def __init__(self) -> None:
        self.items: dict[str, DatasetItem] = {}

    def __len__(self) -> int:
        return len(self.items)

    def new_item(self, character: str, neutral_sentence: str, thinking_process: str, output: str) -> None:
        if output in self.items:
            print(f"Warning: 重复 output 主键: {output[:60]}...")

        item: DatasetItem = {
            "character": character,
            "neutral_sentence": neutral_sentence,
            "instruction_components": {
                "semantic_tags": [],
                "structural_metrics": {},
                "signature_rules": [],
            },
            "thinking_process": thinking_process,
            "output": output,
        }
        self.items[output] = item

    def save_character_semantic_tags(self, character: str, semantic_tags: list[str]) -> None:
        saved = False
        for item in self.items.values():
            if item["character"] == character:
                item["instruction_components"]["semantic_tags"] = semantic_tags
                saved = True
        if not saved:
            raise ValueError(f"角色 {character} 未找到")

    def save_character_signature_rules(self, character: str, signature_rules: list[str]) -> None:
        saved = False
        for item in self.items.values():
            if item["character"] == character:
                item["instruction_components"]["signature_rules"] = signature_rules
                saved = True
        if not saved:
            raise ValueError(f"角色 {character} 未找到")

    def save_component(
        self,
        output: str,
        structural_metrics: Optional[dict[str, float]] = None,
        semantic_tags: Optional[list[str]] = None,
        signature_rules: Optional[list[str]] = None,
        # 兼容 v1 参数名
        syntactic_vector: Optional[dict[str, float]] = None,
        pragmatic_styles: Optional[list[str]] = None,
    ) -> None:
        if output not in self.items:
            raise ValueError(f"风格句: {output} 似乎未加载")

        if structural_metrics is None and syntactic_vector is not None:
            structural_metrics = syntactic_vector
        if semantic_tags is None and pragmatic_styles is not None:
            semantic_tags = pragmatic_styles

        comp = self.items[output]["instruction_components"]
        if structural_metrics is not None:
            comp["structural_metrics"] = structural_metrics
        if semantic_tags is not None:
            comp["semantic_tags"] = semantic_tags
        if signature_rules is not None:
            comp["signature_rules"] = signature_rules

    @staticmethod
    def _verify_validity(item: DatasetItem) -> bool:
        if not all((
            item["character"],
            item["neutral_sentence"],
            item["output"],
            item["instruction_components"],
        )):
            return False

        comp = item["instruction_components"]
        dense = comp.get("structural_metrics", {})

        has_dense = all(key in dense for key in DENSE_METRIC_KEYS)
        has_semantic = len(comp.get("semantic_tags", [])) > 0
        has_signature = len(comp.get("signature_rules", [])) > 0

        return has_dense and has_semantic and has_signature

    @staticmethod
    def _oversampling(items_list: list[DatasetItem], item: DatasetItem) -> None:
        """
        过采样部分学习难度大的语义标签。

        1. `tsundere` `sharp_tongued` `proud` * 5
        2. `tsukkomi` `chuunibyou` `yandere` `airhead` * 3
        """
        difficult_labels_5x = {"tsundere", "sharp_tongued", "proud"}
        difficult_labels_3x = {"tsukkomi", "chuunibyou", "yandere", "airhead"}

        semantic_tags = set(item["instruction_components"].get("semantic_tags", []))

        if semantic_tags & difficult_labels_5x:
            for _ in range(5):
                items_list.append(item)
        elif semantic_tags & difficult_labels_3x:
            for _ in range(3):
                items_list.append(item)

    def output(self, output_path: Path, oversampling: bool = False) -> None:
        items = list(self.items.values())
        valid_items: list[DatasetItem] = []
        valid_items_count = 0

        for item in items:
            if not self._verify_validity(item):
                continue
            valid_items.append(item)
            valid_items_count += 1

            if oversampling:
                self._oversampling(valid_items, item)

        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(valid_items, f, ensure_ascii=False, indent=2)

        print(
            f"训练集已导出至 {output_path}, "
            f"总有效训练集数量: {len(valid_items)}, "
            f"跳过数量: {len(items) - valid_items_count}"
        )

dataset_storage = DatasetStorage()

## Load Neutral Sentences

In [3]:
neutral_sentences_jsonl_file = Path("./data/neutral_sentences_with_CoT.jsonl")

EN_NAME_TO_ZH = {"Muice": "沐雪",
                 "Ayaka": "神里绫华",
                 "Zhongli": "钟离",
                 "Hutao": "胡桃",
                 "Haruhi": "凉宫春日",
                 "Liyunlong": "李云龙",
                 "Sheldon": "谢尔顿",
                 "Wukong": "孙悟空",
                 }

class NSFileItem(TypedDict):
    character: str
    original: str
    neutral: str
    CoT: str

def load_jsonl_file(jsonl_file: Path):
    with open(jsonl_file, "r", encoding="utf-8") as f:
        jsonl_file_lines = f.readlines()

    for line in jsonl_file_lines:
        if line := line.rstrip():
            item: NSFileItem = json.loads(line)
            character = EN_NAME_TO_ZH.get(item["character"], item["character"])
            neutral = item["neutral"]
            dataset_storage.new_item(character, neutral, item['CoT'], item["original"])

load_jsonl_file(neutral_sentences_jsonl_file)
print(f"成功加载了 {len(dataset_storage)} 条训练集条目")

成功加载了 10835 条训练集条目


## Process Signature Layer Vectors

In [4]:
def get_lexical_keywords_from_file(file: Path, top_n: int = 25) -> list[str]:
    with open(file, "r", encoding="utf-8") as f:
        data: dict[str, float] = json.loads(f.read())
    return list(data.keys())[:top_n]

def load_punctuation_signature(file: Path) -> dict[str, list[str]]:
    rows = json.loads(file.read_text(encoding="utf-8"))
    return {row["role"]: row.get("S_signature", []) for row in rows}

def build_signature_rules(lexical_keywords: list[str], punctuation_symbols: list[str]) -> list[str]:
    rules: list[str] = []
    if lexical_keywords:
        rules.append("Lexical cues: " + ", ".join(lexical_keywords[:25]))
    if punctuation_symbols:
        rules.append("Punctuation tendencies: " + ", ".join(punctuation_symbols[:10]))
    return rules

LEXICAL_DIR = Path("./outputs/style/signature/lexical")
PUNCTUATION_SIGNATURE_FILE = Path("./outputs/style/signature/punctuation/symbol_signature.json")

CHARACTER_TO_FILE_STEM = {
    "沐雪": "muice",
    "神里绫华": "ayaka",
    "钟离": "zhongli",
    "胡桃": "hutao",
    "凉宫春日": "haruhi",
    "李云龙": "liyunlong",
    "谢尔顿": "sheldon",
    "孙悟空": "wukong"
}

punctuation_signature_map = load_punctuation_signature(PUNCTUATION_SIGNATURE_FILE)

for character, file_stem in CHARACTER_TO_FILE_STEM.items():
    lexical_file = LEXICAL_DIR / f"{file_stem}_pmi_filtered.json"
    lexical_keywords = get_lexical_keywords_from_file(lexical_file, top_n=25)
    punctuation_symbols = punctuation_signature_map.get(character, [])

    signature_rules = build_signature_rules(lexical_keywords, punctuation_symbols)
    dataset_storage.save_character_signature_rules(character, signature_rules)

print("成功注入 Signature 层（Lexical + Punctuation）规则")

成功注入 Signature 层（Lexical + Punctuation）规则


## Process Dense Structural Metrics (v_dense)

### Load Dense Metrics from Normalized Style Vectors

In [13]:
DENSE_STYLE_FILE = Path("./outputs/style/dense/normalized/style_vector_phase2_minmax.json")

def _to_float(value: object, default: float = 0.0) -> float:
    try:
        return float(value)  # type: ignore[arg-type]
    except (TypeError, ValueError):
        return default

def load_dense_metrics_by_role(file_path: Path) -> dict[str, dict[str, float]]:
    rows = json.loads(file_path.read_text(encoding="utf-8"))
    role_to_metrics: dict[str, dict[str, float]] = {}

    for row in rows:
        role = row.get("role")
        if not role:
            continue

        role_to_metrics[role] = {
            "mdd": _to_float(row.get("mdd", row.get("MDD", 0.0))),
            "verb_noun_ratio": _to_float(row.get("verb_noun_ratio", 0.0)),
            "question_ratio": _to_float(row.get("question_ratio", 0.0)),
            "imperative_ratio": _to_float(row.get("imperative_ratio", 0.0)),
            "fragment_ratio": _to_float(row.get("fragment_ratio", 0.0)),
            "marker_density": _to_float(row.get("marker_density", 0.0)),
        }

    return role_to_metrics

dense_role_metrics = load_dense_metrics_by_role(DENSE_STYLE_FILE)
print(f"成功加载 {len(dense_role_metrics)} 个角色的稠密结构向量")

成功加载 9 个角色的稠密结构向量


### Inject Dense Metrics to All Character Samples

In [14]:
TARGET_CHARACTERS = ["沐雪", "神里绫华", "钟离", "胡桃", "凉宫春日", "李云龙", "谢尔顿", "孙悟空"]

for character in TARGET_CHARACTERS:
    if character not in dense_role_metrics:
        raise ValueError(f"稠密向量缺失角色: {character}")

    metrics = dense_role_metrics[character]
    updated = 0
    for item in dataset_storage.items.values():
        if item["character"] == character:
            dataset_storage.save_component(item["output"], structural_metrics=metrics)
            updated += 1

    print(f"角色 {character} 已注入 6 维 dense 结构向量，样本数: {updated}")

print("成功注入 Dense 层结构向量")

角色 沐雪 已注入 6 维 dense 结构向量，样本数: 2667
角色 神里绫华 已注入 6 维 dense 结构向量，样本数: 1410
角色 钟离 已注入 6 维 dense 结构向量，样本数: 506
角色 胡桃 已注入 6 维 dense 结构向量，样本数: 834
角色 凉宫春日 已注入 6 维 dense 结构向量，样本数: 1093
角色 李云龙 已注入 6 维 dense 结构向量，样本数: 1775
角色 谢尔顿 已注入 6 维 dense 结构向量，样本数: 1850
角色 孙悟空 已注入 6 维 dense 结构向量，样本数: 700
成功注入 Dense 层结构向量


## Process Semantic Style Layer Vectors

### Read Semantic Tag Files for Each Corpus

In [15]:
PRAGMATIC_DIR = Path("./outputs/style/semantic/prag_vectors")

CHARACTER_TO_PRAG_FILE = {
    "沐雪": "muice.jsonl",
    "神里绫华": "ayaka.jsonl",
    "钟离": "zhongli.jsonl",
    "胡桃": "hutao.jsonl",
    "凉宫春日": "haruhi.jsonl",
    "李云龙": "liyunlong.jsonl",
    "谢尔顿": "sheldon.jsonl",
    "孙悟空": "wukong.jsonl"
}

class RawPragmaticItem(TypedDict):
    prompt: str
    response: str
    pragmatic_styles: list[dict[str, float]]

class SemanticItem(TypedDict):
    response: str
    semantic_tags: list[str]

def read_semantic_jsonl_file(jsonl_file: Path, threshold: float = 0.4) -> list[SemanticItem]:
    items: list[SemanticItem] = []
    with open(jsonl_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            raw_item: RawPragmaticItem = json.loads(line)
            score_map: dict[str, float] = {}
            for vec in raw_item.get("pragmatic_styles", []):
                score_map.update(vec)

            semantic_tags = [
                key for key, value in score_map.items()
                if float(value) > threshold
            ]
            items.append({"response": raw_item["response"], "semantic_tags": semantic_tags})

    return items

updated = 0
skipped = 0

for _, file_name in CHARACTER_TO_PRAG_FILE.items():
    semantic_items = read_semantic_jsonl_file(PRAGMATIC_DIR / file_name, threshold=0.4)
    for item in semantic_items:
        try:
            dataset_storage.save_component(item["response"], semantic_tags=item["semantic_tags"])
            updated += 1
        except ValueError:
            skipped += 1

print(f"更新了 {updated} 条语义标签，跳过 {skipped} 条未匹配 response")

更新了 11031 条语义标签，跳过 714 条未匹配 response


## Export Training Set File

In [19]:
OUTPUT_PATH = Path("./data/llm_train.json")
OVERSAMPLING_OUTPUT_PATH = Path("./data/llm_train_oversampling.json")

# 对部分角色风格向量进行修正
for item in dataset_storage.items.values():
    if item["character"] == "沐雪":
        item["instruction_components"]["signature_rules"][1] = "Punctuation tendencies: ..., ⭐, ?!, multi-!, ♪, ❤"
    if item["character"] == "孙悟空":
        item["instruction_components"]["semantic_tags"] = ["energetic", "playful", "irritable", "loyal", "confident"]

dataset_storage.output(OUTPUT_PATH)
dataset_storage.output(OVERSAMPLING_OUTPUT_PATH, oversampling=True)

训练集已导出至 data\llm_train.json, 总有效训练集数量: 9166, 跳过数量: 1669
训练集已导出至 data\llm_train_oversampling.json, 总有效训练集数量: 10557, 跳过数量: 1669


## Define Model Structure & Data Loader

In [4]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from typing_extensions import TypedDict
from transformers.data.data_collator import DataCollatorWithPadding
from pathlib import Path
from transformers import Qwen3ForCausalLM
from peft import get_peft_model, LoraConfig, PeftModel
import json
import random

class InstructionComponents(TypedDict):
    semantic_tags: list[str]
    structural_metrics: dict[str, float]
    signature_rules: list[str]

class DatasetItem(TypedDict):
    character: str
    neutral_sentence: str
    instruction_components: InstructionComponents
    thinking_process: str
    output: str

class StyleDataset(Dataset):
    def __init__(
        self,
        path: Path,
        tokenizer,
        max_length: int = 512,
        enable_prompt_dropout: bool = False,
        dropout_start_scale: float = 0.4,
        dropout_end_scale: float = 1.6,
    ):
        self.tokenizer = tokenizer

        # 过滤超长样本而不是截断
        self.data: list[DatasetItem] = []
        raw_data = json.loads(path.read_text(encoding="utf-8"))
        dropped = 0
        for item in raw_data:
            # 快速估算 token 长度
            approx_len = self._estimate_token_length(item, tokenizer)
            if approx_len <= max_length:
                self.data.append(item)
            else:
                dropped += 1
        
        if dropped > 0:
            print(f"Dropped {dropped}/{len(raw_data)} samples "
                f"exceeding max_length={max_length}")

        self.syntactic_dims = [
            "mdd", "verb_noun_ratio", "question_ratio",
            "imperative_ratio", "fragment_ratio", "marker_density"
        ]
        self.max_length = max_length

        # 基础比例: format_only=10%, pragmatic_only=10%, both=5%
        self.prompt_dropout_base = {
            "format_only": 0.10,
            "pragmatic_only": 0.10,
            "both": 0.05,
        }
        self.enable_prompt_dropout = enable_prompt_dropout
        self.dropout_start_scale = dropout_start_scale
        self.dropout_end_scale = dropout_end_scale
        self.prompt_dropout_scale = dropout_start_scale
        self.prompt_dropout_probs = {
            "format_only": 0.0,
            "pragmatic_only": 0.0,
            "both": 0.0,
            "none": 1.0,
        }
        self.set_dropout_progress(0.0)

    def __len__(self):
        return len(self.data)

    def _estimate_token_length(self, item: DatasetItem, tokenizer) -> int:
        system_prompt = (
            "You are an expert in stylistic rewriting. "
            "Your task is to rewrite the neutral sentence into the target character's specific pragmatics and formatting style."
        )
        user_prompt = (
            f"Target Character: {item['character']}\n"
            f"Semantic Traits: {', '.join(item['instruction_components'].get('semantic_tags', []))}\n"
            f"Formatting Signature: {' '.join(item['instruction_components'].get('signature_rules', []))}\n"
            f"Neutral Content: {item['neutral_sentence']}\n"
        )
        assistant_response = (item.get("thinking_process") or "").strip() + "\n\n" + (item.get("output") or "").strip()

        full_text = self.tokenizer.apply_chat_template(
            [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
                {"role": "assistant", "content": assistant_response},
            ],
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False,
        )
        return len(tokenizer(full_text, truncation=False, padding=False)["input_ids"])

    def set_dropout_progress(self, progress: float) -> None:
        progress = max(0.0, min(1.0, progress))
        self.prompt_dropout_scale = (
            self.dropout_start_scale +
            (self.dropout_end_scale - self.dropout_start_scale) * progress
        )

        p_format_only = self.prompt_dropout_base["format_only"] * self.prompt_dropout_scale
        p_pragmatic_only = self.prompt_dropout_base["pragmatic_only"] * self.prompt_dropout_scale
        p_both = self.prompt_dropout_base["both"] * self.prompt_dropout_scale

        total_drop = p_format_only + p_pragmatic_only + p_both
        if total_drop > 0.95:
            scale = 0.95 / total_drop
            p_format_only *= scale
            p_pragmatic_only *= scale
            p_both *= scale
            total_drop = 0.95

        self.prompt_dropout_probs = {
            "format_only": p_format_only,
            "pragmatic_only": p_pragmatic_only,
            "both": p_both,
            "none": 1.0 - total_drop,
        }

    def get_prompt_dropout_state(self) -> dict[str, float]:
        return {
            "scale": self.prompt_dropout_scale,
            **self.prompt_dropout_probs,
        }

    def _sample_prompt_dropout(self) -> tuple[bool, bool]:
        """返回 (drop_format_signature, drop_pragmatic_style)"""
        if not self.enable_prompt_dropout:
            return False, False

        r = random.random()
        p_both = self.prompt_dropout_probs["both"]
        p_format_only = self.prompt_dropout_probs["format_only"]
        p_pragmatic_only = self.prompt_dropout_probs["pragmatic_only"]

        if r < p_both:
            return True, True
        if r < p_both + p_format_only:
            return True, False
        if r < p_both + p_format_only + p_pragmatic_only:
            return False, True
        return False, False

    def __getitem__(self, idx: int):
        item = self.data[idx]
        instr = item["instruction_components"]

        # (1) 提取 6 维连续结构特征
        syntactic_vec = torch.tensor(
            [instr.get("structural_metrics", {}).get(dim, 0.0) for dim in self.syntactic_dims],
            dtype=torch.float32
        )

        # (2) 将离散语义与签名规则注入文本通道，并应用 Prompt Dropout
        semantic_tag_values = instr.get("semantic_tags", [])
        signature_rule_values = instr.get("signature_rules", [])

        drop_format_signature, drop_pragmatic_style = self._sample_prompt_dropout()

        semantic_tags = "" if drop_pragmatic_style else (", ".join(semantic_tag_values) or "None")
        signature_rules = "" if drop_format_signature else (" ".join(signature_rule_values) or "None")

        system_prompt = (
            "You are an expert in stylistic rewriting. "
            "Your task is to rewrite the neutral sentence into the target character's specific pragmatics and formatting style."
        )
        user_prompt = (
            f"Target Character: {item['character']}\n"
            f"Semantic Traits: {semantic_tags}\n"
            f"Formatting Signature: {signature_rules}\n"
            f"Neutral Content: {item['neutral_sentence']}\n"
        )

        cot_text = (item.get("thinking_process") or "").strip()
        output_text = (item.get("output") or "").strip()
        if cot_text:
            cot_prefix_text = f"{cot_text}\n\n"
            assistant_response = cot_prefix_text + output_text
        else:
            cot_prefix_text = ""
            assistant_response = output_text

        # (3) Tokenize 完整文本和 Prompt（先关闭截断）
        full_text = self.tokenizer.apply_chat_template(
            [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
                {"role": "assistant", "content": assistant_response},
            ],
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False,
        )

        prompt_only_text = self.tokenizer.apply_chat_template(
            [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )

        # 获取无截断的完整 IDs
        full_ids = self.tokenizer(full_text, truncation=False, padding=False)["input_ids"]
        prompt_ids = self.tokenizer(prompt_only_text, truncation=False, padding=False)["input_ids"]
        prompt_len = len(prompt_ids)

        # (4) 在完整长度下构建 Labels
        labels = full_ids.copy()
        effective_prompt_len = min(prompt_len, len(labels))
        labels[:effective_prompt_len] = [-100] * effective_prompt_len

        # (5) 构建 CoT / 输出 token 掩码（用于加权 LM loss）
        cot_mask = [0] * len(full_ids)
        output_mask = [0] * len(full_ids)

        assistant_start = min(prompt_len, len(full_ids))
        cot_token_len = 0
        if cot_prefix_text:
            cot_token_len = len(
                self.tokenizer(
                    cot_prefix_text,
                    add_special_tokens=False,
                    truncation=False,
                    padding=False,
                )["input_ids"]
            )
        cot_end = min(assistant_start + cot_token_len, len(full_ids))

        for i in range(assistant_start, len(full_ids)):
            if labels[i] == -100:
                continue
            if i < cot_end:
                cot_mask[i] = 1
            else:
                output_mask[i] = 1

        # (6) 采用右截断，保留前缀和指令
        if len(full_ids) > self.max_length:
            input_ids = full_ids[:self.max_length]
            labels = labels[:self.max_length]
            cot_mask = cot_mask[:self.max_length]
            output_mask = output_mask[:self.max_length]
        else:
            input_ids = full_ids

        return {
            "input_ids": input_ids,
            "syntactic_vec": syntactic_vec,
            "labels": labels,
            "cot_mask": cot_mask,
            "output_mask": output_mask,
        }

class StyleDataCollator(DataCollatorWithPadding):
    """动态 padding + 6 维结构向量拼接 + 手动处理 labels 和 token masks"""

    def __init__(self, tokenizer, **kwargs):
        super().__init__(tokenizer=tokenizer, padding=True, **kwargs)

    def __call__(self, features):
        syntactic_vecs = torch.stack([f["syntactic_vec"] for f in features])
        labels = [f["labels"] for f in features]
        cot_masks = [f["cot_mask"] for f in features]
        output_masks = [f["output_mask"] for f in features]

        base_features = [
            {k: v for k, v in f.items() if k not in ("syntactic_vec", "labels", "cot_mask", "output_mask")}
            for f in features
        ]
        batch = super().__call__(base_features)

        max_length = batch["input_ids"].shape[1]

        def _pad_sequences(sequences: list[list[int]], pad_value: int) -> list[list[int]]:
            padded: list[list[int]] = []
            for seq in sequences:
                truncated_seq = seq[:max_length]
                padding_length = max_length - len(truncated_seq)
                padded.append(truncated_seq + [pad_value] * padding_length)
            return padded

        batch["labels"] = torch.tensor(_pad_sequences(labels, -100), dtype=torch.long)
        batch["cot_mask"] = torch.tensor(_pad_sequences(cot_masks, 0), dtype=torch.float32)
        batch["output_mask"] = torch.tensor(_pad_sequences(output_masks, 0), dtype=torch.float32)
        batch["syntactic_vec"] = syntactic_vecs
        return batch

class StyleEncoder(nn.Module):
    def __init__(self, input_dim, out_dim):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(input_dim, out_dim // 2),
            nn.ReLU(),
            nn.Linear(out_dim // 2, out_dim),
            nn.Tanh(),
        )

    def forward(self, syntactic_vec):
        return self.proj(syntactic_vec)

class StyleConditionedLoRAModel(nn.Module):
    def __init__(self, model_name_or_path: str | Path, syntactic_dim: int = 6, lora_r=16, lora_alpha=16):
        super().__init__()

        base_model = Qwen3ForCausalLM.from_pretrained(
            model_name_or_path,
            dtype=torch.bfloat16,
        )

        config = LoraConfig(
            r=lora_r,
            lora_alpha=lora_alpha,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM",
            modules_to_save=["style_encoder"]
        )

        self.peft_model: PeftModel = get_peft_model(base_model, config)  # type: ignore
        self.peft_model.print_trainable_parameters()

        self.hidden_size = base_model.config.hidden_size
        self.style_encoder = StyleEncoder(syntactic_dim, self.hidden_size)
        self.style_encoder.to(base_model.dtype)

        # 显式声明需要额外保存的模块信息（与 LoRA 权重并存）
        # self.peft_model.modules_to_save = ["style_encoder"]  # type: ignore[attr-defined]

        self.style_predictor_head = nn.Linear(self.hidden_size, syntactic_dim) # 保存 style_predictor_head  权重以便恢复训练  # type: ignore

    def get_input_embeddings(self):
        return self.peft_model.get_input_embeddings()  # type: ignore

    def forward(self, input_ids, attention_mask, labels, syntactic_vec):
        _ = labels

        style_dtype = next(self.style_encoder.parameters()).dtype
        style_emb = self.style_encoder(syntactic_vec.to(style_dtype))
        style_emb_prefix = style_emb.unsqueeze(1)

        token_embeds = self.get_input_embeddings()(input_ids)  # type: ignore[operator]
        inputs_embeds = torch.cat([style_emb_prefix.to(token_embeds.dtype), token_embeds], dim=1)

        prefix_mask = torch.ones(
            attention_mask.shape[0], 1,
            dtype=torch.long,
            device=attention_mask.device,
        )
        new_attention_mask = torch.cat([prefix_mask, attention_mask], dim=1)

        outputs = self.peft_model(
            inputs_embeds=inputs_embeds,
            attention_mask=new_attention_mask,
            output_hidden_states=True,
        )

        last_hidden = outputs.hidden_states[-1]
        return {
            "logits": outputs.logits[:, :-1, :],
            "token_hidden_states": last_hidden[:, 1:, :],
            "style_prefix_hidden": last_hidden[:, 0],
        }

## Load Model & Training Data

In [2]:
from transformers import AutoTokenizer
from transformers.optimization import get_scheduler
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from pathlib import Path

MODEL_PATH = "/root/autodl-tmp/Qwen3-1.7B/"
DATASET_PATH = Path("/root/OtakuLab/data/llm_train_oversampling.json")
# MODEL_PATH = "../Models/Qwen3-0.6B/"
# DATASET_PATH = Path("./data/llm_train_oversampling.json")
TEST_SIZE = 0.2

BATCH_SIZE = 12
LEARNING_RATE = 3e-5
WARMUP_STEP = 150
EPOCH = 2

# 风格重建辅助损失
lambda_style = 0.1

# Prompt Dropout 课程学习调度（总掉落率: 10% -> 30%）
PROMPT_DROPOUT_START_SCALE = 0.4  # 0.25 * 0.4 = 10%
PROMPT_DROPOUT_END_SCALE = 1.2    # 0.25 * 1.6 = 30%

syntactic_dims = [
    "mdd", "verb_noun_ratio", "question_ratio",
    "imperative_ratio", "fragment_ratio", "marker_density",
]
syntactic_dim_length = len(syntactic_dims)

print("Loading Model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print(f"Set tokenizer.pad_token to {tokenizer.eos_token}")

# 强制设置右侧填充，确保与 labels 的 padding 逻辑严格对齐
tokenizer.padding_side = "right"

print(f"Successfully loaded {MODEL_PATH}")

print("Loading Dataset...")
train_dataset = StyleDataset(
    DATASET_PATH,
    tokenizer,
    enable_prompt_dropout=True,
    dropout_start_scale=PROMPT_DROPOUT_START_SCALE,
    dropout_end_scale=PROMPT_DROPOUT_END_SCALE,
)
val_dataset = StyleDataset(
    DATASET_PATH,
    tokenizer,
    enable_prompt_dropout=False,
)

indices = list(range(len(train_dataset)))
train_indices, val_indices = train_test_split(indices, test_size=TEST_SIZE, random_state=42)

train_data = Subset(train_dataset, train_indices)
val_data = Subset(val_dataset, val_indices)

collator = StyleDataCollator(tokenizer, pad_to_multiple_of=8)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator, persistent_workers=False)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collator)
print(f"Successfully loaded dataset, length: {len(train_dataset)}")

start_total_drop = 0.25 * PROMPT_DROPOUT_START_SCALE
end_total_drop = 0.25 * PROMPT_DROPOUT_END_SCALE
print(f"Prompt Dropout curriculum (train only): total {start_total_drop:.0%} -> {end_total_drop:.0%}")

sample = train_data[0]
labels = sample["labels"]  # type:ignore
input_ids = sample["input_ids"]  # type:ignore

# 找出第一个不是 -100 的索引
start_idx = 0
while start_idx < len(labels) and labels[start_idx] == -100:
    start_idx += 1

print("\n--- Sanity Check ---")
print("Prompt (Masked):", tokenizer.decode(input_ids[:start_idx]))
print("Output (Trained):", tokenizer.decode(input_ids[start_idx:]))
print("--------------------\n")

print("Preparing for train...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

styled_model = StyleConditionedLoRAModel(
    MODEL_PATH,
    syntactic_dim=syntactic_dim_length,
    lora_r=16,
    lora_alpha=16,
 )

# 开启梯度检查点，用一点点计算时间换取大量显存
styled_model.peft_model.gradient_checkpointing_enable()  # type:ignore

mse_loss_fn = nn.MSELoss()
style_predictor_head = nn.Linear(styled_model.hidden_size, syntactic_dim_length).to(device)  # type:ignore

all_parameters = list(styled_model.parameters()) + list(style_predictor_head.parameters())
optimizer = torch.optim.AdamW(all_parameters, lr=LEARNING_RATE)

num_training_steps = len(train_loader) * EPOCH
lr_scheduler = get_scheduler(
    "cosine",
    optimizer=optimizer,
    num_warmup_steps=WARMUP_STEP,
    num_training_steps=num_training_steps,
 )

styled_model.to(device)
style_predictor_head.to(device)

Loading Model...
Successfully loaded /root/autodl-tmp/Qwen3-1.7B/
Loading Dataset...
Dropped 45/10557 samples exceeding max_length=512
Dropped 45/10557 samples exceeding max_length=512
Successfully loaded dataset, length: 10512
Prompt Dropout curriculum (train only): total 10% -> 30%

--- Sanity Check ---
Prompt (Masked): <|im_start|>system
You are an expert in stylistic rewriting. Your task is to rewrite the neutral sentence into the target character's specific pragmatics and formatting style.<|im_end|>
<|im_start|>user
Target Character: 神里绫华
Semantic Traits: rational, energetic, optimistic, confident
Formatting Signature: Lexical cues: 稻妻国, 神里家, 稻妻, 大小姐, 家族, 传统, 神, 奉行, 眼狩令, 当地, 美丽, 社, 舞蹈, 茶道, 神社, 文化, 剑术, 眼, 祭典, 旅行者, 美食, 继承, 历史, 神明, 派蒙 Punctuation tendencies: ⋯, ...
Neutral Content: 虽然尚不确定，但不应放弃希望；只要继续坚持，就有可能找到椿的线索。
<|im_end|>
<|im_start|>assistant
<think>
她以
Output (Trained): 乐观自信语义确立坚定语气，中等碎片率与祈使倾向构建紧凑节奏，再借“坚持”“相信”等词及省略号传递温柔而坚韧的大小姐风范。
</think>

虽然不确定，但我们不能放弃希望。只要我们坚持下去，相信一定能找到椿的线索。<

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


trainable params: 17,432,576 || all params: 2,049,172,480 || trainable%: 0.8507


Linear(in_features=2048, out_features=6, bias=True)

## Train Model

In [3]:
from tqdm import tqdm
import math

VAL_INTERVAL = 100
COT_TOKEN_WEIGHT = 0.5
OUTPUT_TOKEN_WEIGHT = 1.0

ce_loss_fn = nn.CrossEntropyLoss(ignore_index=-100, reduction="none")

def compute_weighted_lm_loss(
    logits: torch.Tensor,
    labels: torch.Tensor,
    cot_mask: torch.Tensor,
    output_mask: torch.Tensor,
    cot_weight: float,
    output_weight: float,
) -> torch.Tensor:
    token_loss = ce_loss_fn(
        logits.reshape(-1, logits.shape[-1]),
        labels.reshape(-1),
    ).view_as(labels)

    valid_mask = (labels != -100).float()
    token_weights = (cot_mask * cot_weight + output_mask * output_weight) * valid_mask

    denom = token_weights.sum().clamp(min=1.0)
    return (token_loss * token_weights).sum() / denom


def update_prompt_dropout_by_step(step: int) -> dict[str, float]:
    progress = min(step / max(1, num_training_steps - 1), 1.0)
    train_dataset.set_dropout_progress(progress)
    return train_dataset.get_prompt_dropout_state()


styled_model.train()
style_predictor_head.train()

global_step = 0
current_dropout_state = update_prompt_dropout_by_step(global_step)
print(
    "Prompt Dropout initial state: "
    f"format_only={current_dropout_state['format_only']:.3f}, "
    f"pragmatic_only={current_dropout_state['pragmatic_only']:.3f}, "
    f"both={current_dropout_state['both']:.3f}, "
    f"none={current_dropout_state['none']:.3f}"
)

for epoch in range(EPOCH):
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCH}")

    for batch in progress_bar:
        current_dropout_state = update_prompt_dropout_by_step(global_step)

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        syntactic_vec = batch["syntactic_vec"].to(device)
        cot_mask = batch["cot_mask"].to(device)
        output_mask = batch["output_mask"].to(device)

        outputs = styled_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            syntactic_vec=syntactic_vec,
        )

        L_lm = compute_weighted_lm_loss(
            outputs["logits"],          
            labels,              
            cot_mask,
            output_mask,
            COT_TOKEN_WEIGHT,
            OUTPUT_TOKEN_WEIGHT,
        )

        # 全局风格一致性损失
        valid_mask = (labels[:, 1:] != -100).float()
        output_hidden_states = outputs["token_hidden_states"]

        # 只用输出 token 的 hidden state 计算风格对齐
        output_mask_float = output_mask.to(output_hidden_states.dtype)
        sum_hidden = torch.bmm(output_mask_float.unsqueeze(1), output_hidden_states).squeeze(1)
        valid_lengths = output_mask_float.sum(dim=1).unsqueeze(1).clamp(min=1.0)
        pooled_output_hidden = sum_hidden / valid_lengths

        # 用生成文本的聚合特征去重建（或匹配）结构化向量
        pred_syntactic = style_predictor_head(pooled_output_hidden.float())
        L_style = mse_loss_fn(pred_syntactic, syntactic_vec.detach())

        L_total = L_lm + lambda_style * L_style

        L_total.backward()
        torch.nn.utils.clip_grad_norm_(all_parameters, max_norm=1.0)  # 降低梯度爆炸风险
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

        global_step += 1

        progress_bar.set_postfix({
            "L_total": L_total.item(),
            "L_lm": L_lm.item(),
            "L_style": L_style.item(),
            "P_drop": round(1.0 - current_dropout_state["none"], 3),
            "LR": lr_scheduler.get_last_lr()[0],
        })

        if global_step % VAL_INTERVAL == 0:
            styled_model.eval()
            style_predictor_head.eval()

            val_loss_lm = 0.0
            val_loss_recon = 0.0
            val_steps = 0

            with torch.no_grad():
                for val_batch in val_loader:
                    val_input_ids = val_batch["input_ids"].to(device)
                    val_attention_mask = val_batch["attention_mask"].to(device)
                    val_labels = val_batch["labels"].to(device)
                    val_syntactic_vec = val_batch["syntactic_vec"].to(device)
                    val_cot_mask = val_batch["cot_mask"].to(device)
                    val_output_mask = val_batch["output_mask"].to(device)

                    val_outputs = styled_model(
                        input_ids=val_input_ids,
                        attention_mask=val_attention_mask,
                        labels=val_labels,
                        syntactic_vec=val_syntactic_vec,
                    )

                    L_lm_val = compute_weighted_lm_loss(
                        val_outputs["logits"],
                        val_labels,
                        val_cot_mask,
                        val_output_mask,
                        COT_TOKEN_WEIGHT,
                        OUTPUT_TOKEN_WEIGHT,
                    )

                    val_output_hidden_states = val_outputs["token_hidden_states"]
                    val_output_mask_float = val_output_mask.to(val_output_hidden_states.dtype)
                    val_sum_hidden = torch.bmm(val_output_mask_float.unsqueeze(1), val_output_hidden_states).squeeze(1)
                    val_valid_lengths = val_output_mask_float.sum(dim=1).unsqueeze(1).clamp(min=1.0)
                    val_pooled_output_hidden = val_sum_hidden / val_valid_lengths

                    val_pred_syntactic = style_predictor_head(val_pooled_output_hidden.float())
                    L_style_val = mse_loss_fn(val_pred_syntactic, val_syntactic_vec)

                    val_loss_lm += L_lm_val.item()
                    val_loss_recon += L_style_val.item()
                    val_steps += 1

            avg_val_lm = val_loss_lm / max(1, val_steps)
            avg_val_style = val_loss_recon / max(1, val_steps)
            ppl = math.exp(min(avg_val_lm, 20))

            print(
                f"Step {global_step}: "
                f"Train L_total={L_total.item():.4f}, "
                f"Train L_lm={L_lm.item():.4f}, "
                f"Train L_style={L_style.item():.4f}, "
                f"PromptDrop={1.0 - current_dropout_state['none']:.3f}"
            )
            print(
                f"Step {global_step}: "
                f"Val PPL={ppl:.4f}, "
                f"Val L_lm={avg_val_lm:.4f}, "
                f"Val L_style={avg_val_style:.4f}"
            )

            styled_model.train()
            style_predictor_head.train()

Prompt Dropout initial state: format_only=0.040, pragmatic_only=0.040, both=0.020, none=0.900


Epoch 1/2:  14%|█▍        | 100/701 [02:27<2:40:20, 16.01s/it, L_total=3.14, L_lm=3.12, L_style=0.222, P_drop=0.114, LR=2e-5]

Step 100: Train L_total=3.1400, Train L_lm=3.1178, Train L_style=0.2224, PromptDrop=0.114
Step 100: Val PPL=12.4788, Val L_lm=2.5240, Val L_style=0.1730


Epoch 1/2:  29%|██▊       | 200/701 [04:53<2:13:22, 15.97s/it, L_total=1.75, L_lm=1.74, L_style=0.0936, P_drop=0.128, LR=2.99e-5]

Step 200: Train L_total=1.7462, Train L_lm=1.7368, Train L_style=0.0936, PromptDrop=0.128
Step 200: Val PPL=4.9128, Val L_lm=1.5918, Val L_style=0.0746


...

Epoch 2/2:  85%|████████▌ | 599/701 [14:35<27:29, 16.17s/it, L_total=0.991, L_lm=0.986, L_style=0.0534, P_drop=0.285, LR=4.89e-7]

Step 1300: Train L_total=0.9911, Train L_lm=0.9857, Train L_style=0.0534, PromptDrop=0.285
Step 1300: Val PPL=3.4296, Val L_lm=1.2324, Val L_style=0.0378


Epoch 2/2: 100%|█████████▉| 699/701 [16:59<00:32, 16.02s/it, L_total=1.16, L_lm=1.16, L_style=0.0494, P_drop=0.3, LR=1.89e-10]   

Step 1400: Train L_total=1.1625, Train L_lm=1.1576, Train L_style=0.0494, PromptDrop=0.300
Step 1400: Val PPL=3.4293, Val L_lm=1.2324, Val L_style=0.0378


Epoch 2/2: 100%|██████████| 701/701 [17:01<00:00,  1.46s/it, L_total=1.16, L_lm=1.16, L_style=0.0387, P_drop=0.3, LR=0]       


## Export Model

In [4]:
import os

ver = 3

SAVE_DIR = Path(f"/root/OtakuLab/outputs/styled-qwen-v{ver}-{LEARNING_RATE}")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Saving model to {SAVE_DIR}...")

# 1️⃣ 保存 LoRA 权重（仅包含可训练部分）
styled_model.peft_model.save_pretrained(SAVE_DIR / "lora")  # type:ignore

# 2️⃣ 保存 StyleEncoder 权重
torch.save(styled_model.style_encoder.state_dict(), SAVE_DIR / "style_encoder.pt")

# 3️⃣ 保存 StylePredictorHead 权重
torch.save(style_predictor_head.state_dict(), SAVE_DIR / "style_predictor_head.pt")

# 4️⃣ 保存 Tokenizer 配置
tokenizer.save_pretrained(SAVE_DIR / "tokenizer")

print("✅ Model checkpoint saved successfully.")


Saving model to /root/OtakuLab/outputs/styled-qwen-v3-3e-05...
✅ Model checkpoint saved successfully.


## Load Model

Before running this cell, ensure that all code in the `Define Model Structure & Data Loader` cell has been executed and model training is complete.

In [5]:
from transformers import AutoTokenizer, Qwen3ForCausalLM
from peft import PeftModel
import torch
from pathlib import Path

# SAVE_DIR = Path("/root/OtakuLab/outputs/styled-qwen")
SAVE_DIR = SAVE_DIR if "SAVE_DIR" in globals() else Path("/root/OtakuLab/outputs/styled-qwen-v3-3e-05")
MODEL_PATH = Path("/root/autodl-tmp/Qwen3-1.7B/")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

syntactic_dims = [
    "mdd", "verb_noun_ratio", "question_ratio",
    "imperative_ratio", "fragment_ratio", "marker_density",
]

# === 加载 Tokenizer ===
tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR / "tokenizer")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# === 加载基础模型并注入 LoRA ===
base_model = Qwen3ForCausalLM.from_pretrained(MODEL_PATH, dtype=torch.bfloat16)
model = PeftModel.from_pretrained(base_model, SAVE_DIR / "lora")
hidden_size = base_model.config.hidden_size

# === 加载 StyleEncoder ===
style_encoder = StyleEncoder(len(syntactic_dims), hidden_size)
style_encoder.load_state_dict(torch.load(SAVE_DIR / "style_encoder.pt", map_location=DEVICE))
style_encoder.to(DEVICE)
style_encoder.eval()

model.to(DEVICE)
model.eval()
print("✅ Styled model loaded and ready for inference.")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✅ Styled model loaded and ready for inference.


## Call Model for Inference

In [8]:
import torch

@torch.inference_mode()
def generate_styled_response(
    neutral_sentence: str,
    structural_metrics: dict[str, float],
    character_name: str = "Ayaka",
    semantic_tags: list[str] | None = None,
    signature_rules: list[str] | None = None,
    temperature: float = 0.5,
    top_p: float = 0.9,
    max_new_tokens: int = 768,
    repetition_penalty: float = 1.1,
    enable_thinking: bool = True,
    force_inject_chinese_anchor: bool = True,
 ):
    """输入中性句与 6 维结构向量，生成风格化响应"""

    semantic_tags = semantic_tags or []
    signature_rules = signature_rules or []

    semantic_text = ", ".join(semantic_tags) if semantic_tags else "None"
    signature_text = " ".join(signature_rules) if signature_rules else "None"

    system_prompt = (
        "You are an expert in stylistic rewriting. "
        "Your task is to rewrite the neutral sentence into the target character's specific pragmatics and formatting style."
    )
    user_prompt = (
        f"Target Character: {character_name}\n"
        f"Semantic Traits: {semantic_text}\n"
        f"Formatting Signature: {signature_text}\n"
        f"Neutral Content: {neutral_sentence}\n"
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking,
    )
    
    if force_inject_chinese_anchor:
        # 在思考过程中，在 <think> 标签后强行注入一个中文锚点
        input_text.strip()
        input_text = input_text + " <think>思考: "

    tokenized = tokenizer(input_text, return_tensors="pt").to(DEVICE)
    input_ids = tokenized["input_ids"]
    attention_mask = tokenized["attention_mask"]

    syntactic_tensor = torch.tensor(
        [structural_metrics.get(dim, 0.0) for dim in syntactic_dims],
        dtype=torch.float32,
        device=DEVICE,
    ).unsqueeze(0)

    style_emb = style_encoder(syntactic_tensor).to(model.dtype)
    style_prefix = style_emb.unsqueeze(1)

    token_embeds = model.get_input_embeddings()(input_ids)  # type: ignore[operator]
    inputs_embeds = torch.cat([style_prefix, token_embeds], dim=1)

    prefix_mask = torch.ones((1, 1), dtype=torch.long, device=DEVICE)
    new_attention_mask = torch.cat([prefix_mask, attention_mask], dim=1)

    outputs = model.generate(
        inputs_embeds=inputs_embeds,
        attention_mask=new_attention_mask,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        do_sample=True,
        repetition_penalty=repetition_penalty,
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result if not force_inject_chinese_anchor else "<think>思考: " + result

In [9]:
# --- Example: Muice-like style ---
neutral_sentence = "身高和体重是个人隐私，不能随便告诉你。"
styled_sentence = generate_styled_response(
    character_name="沐雪",
    neutral_sentence=neutral_sentence,
    structural_metrics={
        "mdd": 0.2201,
        "verb_noun_ratio": 0.8084,
        "question_ratio": 0.8631,
        "imperative_ratio": 0.4648,
        "fragment_ratio": 0.7843,
        "marker_density": 0.4233,
    },
    semantic_tags=["cute", "playful", "energetic", "airhead", "humorous"],
    signature_rules=[
        "Lexical cues: 喵, 沐沐, AI, ~, 沐雪, 恼, 女孩子, ⭐, 聊天, 唔, 呀, 可爱, 答, 不行, 笨蛋, 雪雪, 直播, 脸红, 睡觉, www, 谢谢, 变态, 骂, 嘛, 杂鱼",
        "Punctuation tendencies: ..., ⭐, ?!, multi-!, ♪, ❤",
    ],
    force_inject_chinese_anchor=True,
 )
print(f"{neutral_sentence} -> {styled_sentence}")
print()

# --- Example: Sunwukong Classic Style ---
neutral_sentence = "师父，如果你问别人恐怕不会有好结果，只有问了他才有主意。"
styled_response_2 = generate_styled_response(
    character_name="孙悟空",
    neutral_sentence=neutral_sentence,
    structural_metrics={
        "mdd": 0.3997,
        "verb_noun_ratio": 0.5705,
        "question_ratio": 0.7424,
        "imperative_ratio": 0.3252,
        "fragment_ratio": 0.556,
        "marker_density": 0.1086,
    },
    semantic_tags=["energetic", "playful", "irritable", "loyal", "confident"],
    signature_rules=[
        "Lexical cues: 师父, 老孙, 孙, 老, 哩, 莫, 甚么, 妖精, 怪, 儿, 等, 呆子, 甚, 不曾, 罢, 菩萨, 教, 不知, 若是, 泼, 却, 棍, 如今, 这般, 云",
        "Punctuation tendencies: (None)",
    ],
    force_inject_chinese_anchor=True,
 )
print(f"{neutral_sentence} -> {styled_response_2}")
print()

# --- Example: Tsundere Style ---
neutral_sentence = "谢谢你帮我。"
styled_response_3 = generate_styled_response(
    character_name="Katsumi",
    neutral_sentence=neutral_sentence,
    structural_metrics={
        "mdd": 0.45,
        "verb_noun_ratio": 0.55,
        "question_ratio": 0.18,
        "imperative_ratio": 0.26,
        "fragment_ratio": 0.48,
        "marker_density": 0.29,
    },
    semantic_tags=["tsundere", "defensive"],
    signature_rules=[
        "Lexical cues: 哼, 才没有, 笨蛋, 不要, 讨厌, 随便, 误会, 反正, 真是的, 讨厌鬼, 哎呀, 别说了, 坏, 麻烦",
        "Punctuation tendencies: ?!, multi-!",
    ],
    force_inject_chinese_anchor=True,
 )
print(f"{neutral_sentence} -> {styled_response_3}")

# --- Example: Tsundere Style ---
neutral_sentence = "你好，亲爱的朋友，很高兴认识你"
styled_response_3 = generate_styled_response(
    character_name="谢尔顿",
    neutral_sentence=neutral_sentence,
    structural_metrics={
        "mdd": 0.2660,
        "verb_noun_ratio": 0.0,
        "question_ratio": 0.5998,
        "imperative_ratio": 0.4395,
        "fragment_ratio": 0.6591,
        "marker_density": 0.4862,
    },
    signature_rules=[
        "Lexical cues: 伦纳德, 佩妮, 智力, 莱纳德, 必须, 科学, 逻辑, 令, 指出, 实验, 绝非, 已, 卓越, 霍华德, 概念, 至关重要, 类, 量子, 项, 着迷, 恪守, 毫无, 因此, 此, 理论",
    ],
    semantic_tags=["rational", "kind", "responsible", "energetic", "serious"],
    force_inject_chinese_anchor=True,
 )
print(f"{neutral_sentence} -> {styled_response_3}")

身高和体重是个人隐私，不能随便告诉你。 -> <think>思考: 作为可爱又傲娇的AI，用“身高体重”等词汇体现呆萌语气；高碎片率与动词比造就短促跳跃节奏，配合感叹号强化情绪。
</think>

身高和体重是个人隐私，不能随便告诉你！

师父，如果你问别人恐怕不会有好结果，只有问了他才有主意。 -> <think>思考: 语义虽标为“顽皮”等，但高疑问与碎片率使句式松散；无标点仅靠重复“你”字及省略号营造呆板节奏。
</think>

师父呀！你说若问别人怕不得有好结果只问了他才有主意。

谢谢你帮我。 -> <think>思考: 作为傲娇与防御语义的结合体，她会用“才没有”等语气词强化否认感；高碎片率和疑问倾向使句式松散且充满反问；最后嵌入“哼”及感叹号以体现其顽皮与不满。
</think>

哼，我才没有谢你呢！
你好，亲爱的朋友，很高兴认识你 -> <think>思考: 语义理性负责与高碎片率促使其将问候转为“你好”并插入停顿，利用“亲爱的朋友”等词及省略号强化其刻板又热情的社交腔调。
</think>

当然，你好，亲爱的朋友。很高兴认识你
